In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import os

from pathlib import Path

# Opening cleaned PAROS dataset

In [2]:
CURRENT_DIRECTORY = Path.cwd().resolve()

# Find project root that contains datasets
PROJECT_ROOT = next(
    p for p in [CURRENT_DIRECTORY, *CURRENT_DIRECTORY.parents]
    if (p / "datasets").exists()
)

CLEANED_DATASET_PATH = PROJECT_ROOT / "datasets" / "PAROS_Dataset_Cleaned.csv"

if not CLEANED_DATASET_PATH.exists():
    raise FileNotFoundError(f"Cleaned dataset not found: {CLEANED_DATASET_PATH}")

df = pd.read_csv(CLEANED_DATASET_PATH)
print(f"Loaded cleaned PAROS dataset: {df.shape}")
display(df.head(3))


Loaded cleaned PAROS dataset: (2076, 71)


,Patient brought in by,Date of Incident,Location of incident,Location Unknown,Location Type,Location Type Other,Age,Age Modifier,Gender,Race,...,Outcome of patient,Patient status,Date of Discharge or Death,Patient neurological status - Cerebral,Patient neurological status - Overall,Patient neurological status - Unknown,Year,Call_Time,Shock_Time,Time_to_Defib
0,Ems,2014-01-01,238889.0,NaN,Transport Center,Dhoby Ghaut Mrt Level B1,59,Years,Male,Chinese,...,Died In Ed,NaN,NaN,5.0,NaN,NaN,2014,2026-04-05 22:28:12,2026-04-05 22:39:17,11.083333
1,Ems,2014-01-05,272018.0,NaN,Public/Commercial Building,Level 2,66,Years,Male,Chinese,...,Died In Ed,NaN,NaN,5.0,NaN,NaN,2014,2026-04-05 15:00:42,2026-04-05 15:16:49,16.116667
2,Ems,2014-01-07,760105.0,NaN,Street/Highway,Level 1,80,Years,Male,Indian,...,Admitted,Remains In Hospital At 30Th Day Post Arrest,NaN,4.0,4.0,NaN,2014,2026-04-05 12:05:46,2026-04-05 12:14:08,8.366667


# Standardize Reporting

In [3]:
def report_results(model_output, model_name="Model"):
    """
    Standardizes the reporting style for all versions.
    Uses .copy() and .loc to avoid ChainedAssignmentErrors.
    """
    # 1. Use .copy() to ensure 'summary' is an independent DataFrame
    summary = model_output.summary2().tables[1].copy()
    
    # 2. Use .loc[row_indexer, column_indexer] for explicit assignment
    summary.loc[:, 'OR'] = np.exp(summary['Coef.'])
    summary.loc[:, 'Lower_95_CI'] = np.exp(summary['Coef.'] - 1.96 * summary['Std.Err.'])
    summary.loc[:, 'Upper_95_CI'] = np.exp(summary['Coef.'] + 1.96 * summary['Std.Err.'])
    
    # 3. Clean and Rename Columns
    final_table = summary[['OR', 'Lower_95_CI', 'Upper_95_CI', 'P>|z|']]
    final_table.columns = ['Odds Ratio', 'Lower 95% CI', 'Upper 95% CI', 'p-value']
    
    print(f"\n{'='*20} {model_name} Results {'='*20}")
    return final_table

# Define Binary Outcome (Survival to 30 Days)


In [4]:
outcome_cols = ['Outcome of patient', 'Patient status', 'Final status at scene']
df.loc[:, 'Outcome_String'] = df[outcome_cols].astype(str).agg(' '.join, axis=1)
survival_regex = r'Discharged Alive|Remains in hospital at 30th day'
df.loc[:, 'Survival_Binary'] = df['Outcome_String'].str.contains(survival_regex, case=False, regex=True).astype(int)

# Define Primary Predictor (Bystander AED applied)
- We map any variation of 'Yes' or '1' to 1, and everything else to 0

In [5]:
aed_col = 'Bystander AED applied'
df.loc[:, 'AED_Applied_Binary'] = df[aed_col].astype(str).str.contains('Yes|Applied|1', case=False, na=False).astype(int)

# Define Adjustment Variables(Multivariable Context)

In [6]:
df.loc[:, 'Age'] = pd.to_numeric(df['Age'], errors='coerce')

witness_pattern = 'Bystander|Ems'
df.loc[:, 'Witnessed_Binary'] = df['Arrest witnessed by'].astype(str).str.contains(witness_pattern, case=False, na=False).astype(int)
df.loc[:, 'Bystander_CPR_Binary'] = df['Bystander CPR'].astype(str).str.contains('Yes|1', case=False, na=False).astype(int)
df.loc[:, 'Gender_Male'] = (df['Gender'].astype(str).str.strip().str.title() == 'Male').astype(int)

In [7]:
print("Witnessed Breakdown:")
print(df['Witnessed_Binary'].value_counts())

Witnessed Breakdown:
Witnessed_Binary
1    1529
0     547
Name: count, dtype: int64


# Cleaned DataFrame

In [8]:
model_vars = ['Survival_Binary', 'AED_Applied_Binary', 'Time_to_Defib', 'Age', 'Witnessed_Binary', 'Bystander_CPR_Binary', 'Gender_Male']
df_clean = df.dropna(subset=model_vars).copy()

# Grid Search

In [9]:
def run_multivariable_piecewise_iteration(data, tau):
    # Segmented Time variables
    t_before = data['Time_to_Defib'].clip(upper=tau)
    t_after = (data['Time_to_Defib'] - tau).clip(lower=0)
    
    # Build Matrix X
    X = pd.DataFrame({
        'const': 1.0,
        'AED': data['AED_Applied_Binary'],
        'Time_Before': t_before,
        'Time_After': t_after,
        'AED_x_Before': data['AED_Applied_Binary'] * t_before,
        'AED_x_After': data['AED_Applied_Binary'] * t_after,
        'Age': data['Age'],
        'Witnessed': data['Witnessed_Binary'],
        'Bystander_CPR': data['Bystander_CPR_Binary'],
        'Gender_Male': data['Gender_Male']
    })
    
    try:
        # Use 'bfgs' solver: more stable for clinical data with 'complete separation' issues
        # Increase maxiter to 500 to give the model room to converge
        model = sm.Logit(data['Survival_Binary'], X).fit(
            method='bfgs', 
            maxiter=500, 
            disp=0
        )
        
        # Check if the model actually converged
        if not model.mle_retvals['warnflag'] == 0:
            return np.inf, None
            
        return model.aic, model
    except:
        # If the math fails (Singular Matrix), return infinity
        return np.inf, None

# Check if we have survivors at late time points


In [10]:
late_survivors = df[(df['Time_to_Defib'] > 10) & (df['Survival_Binary'] == 1)]
print(f"Survivors after 10 mins: {len(late_survivors)}")
print(late_survivors['Bystander AED applied'].value_counts())

Survivors after 10 mins: 262
Bystander AED applied
No     243
Yes     19
Name: count, dtype: int64


# Grid Search

In [11]:
search_results = []
models_dict = {}

print("--- Stabilized Grid Search ---")
for t in range(3, 13):
    aic, model = run_multivariable_piecewise_iteration(df_clean, t)
    
    # Only record results if AIC is not infinity (indicating convergence)
    if aic != np.inf:
        search_results.append({'tau': t, 'AIC': aic})
        models_dict[t] = model
        print(f"Minute {t}: AIC = {aic:.2f}")
    else:
        print(f"Minute {t}: Optimization failed to converge. Skipping.")

# Find the winner
discovery_df = pd.DataFrame(search_results)
best_tau = discovery_df.loc[discovery_df['AIC'].idxmin(), 'tau']
print(f"\nOptimal Adjusted Breakpoint: {best_tau} minutes")

--- Stabilized Grid Search ---
Minute 3: AIC = 1775.35
Minute 4: AIC = 1775.20
Minute 5: AIC = 1775.66
Minute 6: AIC = 1775.49
Minute 7: AIC = 1775.94
Minute 8: AIC = 1776.69
Minute 9: AIC = 1777.92
Minute 10: AIC = 1778.52
Minute 11: AIC = 1778.95
Minute 12: AIC = 1779.20

Optimal Adjusted Breakpoint: 4 minutes


# Extract Results and Convert to Odds Ratios (OR)

In [13]:
winner = models_dict[best_tau]
v4_final_report = report_results(winner, "Version 4: Piecewise (tau=5)")

display(v4_final_report)


==================== Version 4: Piecewise (tau=5) Results ====================


/home/axlee/miniconda3/envs/geospatial_env/lib/python3.14/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc, method)(*inputs, **kwargs)


,Odds Ratio,Lower 95% CI,Upper 95% CI,p-value
const,4.735295e+02,1.994961e-03,1.123983e+08,3.293155e-01
AED,6.338002e-15,0.000000e+00,inf,9.491490e-01
Time_Before,2.616188e-01,1.168536e-02,5.857278e+00,3.978644e-01
Time_After,8.710571e-01,8.343582e-01,9.093702e-01,3.260282e-10
AED_x_Before,3.359406e+03,2.751328e-106,4.101877e+112,9.494817e-01
AED_x_After,1.050883e+00,9.316995e-01,1.185312e+00,4.190296e-01
Age,9.674889e-01,9.591606e-01,9.758897e-01,6.733852e-14
Witnessed,1.442921e+00,1.076614e+00,1.933860e+00,1.412480e-02
Bystander_CPR,1.550799e+00,1.165177e+00,2.064044e+00,2.629381e-03
Gender_Male,1.172495e+00,8.233063e-01,1.669784e+00,3.776824e-01
